# Routerset `fix27March` Audit

This notebook runs or reuses the exhaustive file-by-file audit for the corrected materialized routerset dataset in `outputs/routerset/fix27March`, then inspects the result with summary tables, plots, and RGB / false-RGB previews.

In [1]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def discover_repo_root() -> Path:
    override = os.environ.get('HYDRANET_REPO_ROOT')
    if override:
        return Path(override).expanduser().resolve()
    for start in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (start / 'pyproject.toml').exists() and (start / 'src').exists():
            return start
    raise RuntimeError('Could not locate repository root')


REPO_ROOT = discover_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from hydranet.routerset_audit import audit_materialized_routerset_dataset


def default_dataset_root() -> Path:
    override = os.environ.get('ROUTERSET_MATERIALIZED_ROOT')
    if override:
        return Path(override).expanduser().resolve()
    return (REPO_ROOT / 'outputs/routerset/fix27March').resolve()


DATASET_ROOT = default_dataset_root()
AUDIT_ROOT = DATASET_ROOT / 'audit'
SUMMARY_PATH = AUDIT_ROOT / 'audit_summary.json'
TILE_AUDIT_PATH = AUDIT_ROOT / 'tile_audit.jsonl'
FAULT_ROWS_PATH = DATASET_ROOT / 'fault_rows_256.jsonl'
MANIFEST_PATH = DATASET_ROOT / 'manifest_256.jsonl'

print('repo_root =', REPO_ROOT)
print('dataset_root =', DATASET_ROOT)
print('audit_root =', AUDIT_ROOT)

repo_root = /shared/home/rdelprete/PythonProjects/hydranet-phisat2
dataset_root = /shared/home/rdelprete/PythonProjects/hydranet-phisat2/outputs/routerset/fix27March
audit_root = /shared/home/rdelprete/PythonProjects/hydranet-phisat2/outputs/routerset/fix27March/audit


In [2]:
if not SUMMARY_PATH.exists() or not TILE_AUDIT_PATH.exists():
    summary = audit_materialized_routerset_dataset(DATASET_ROOT)
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

manifest_rows = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
tile_rows = [json.loads(line) for line in TILE_AUDIT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
fault_rows = [json.loads(line) for line in FAULT_ROWS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]

print(f'manifest rows: {len(manifest_rows):,}')
print(f'tile audit rows: {len(tile_rows):,}')
print(f'fault rows: {len(fault_rows):,}')
print('status counts:', summary['status_counts'])
print('issue counts:', summary['issue_counts'])
print('note counts:', summary['note_counts'])

manifest rows: 24,391
tile audit rows: 24,391
fault rows: 301
status counts: {'ok': 12728, 'warn': 11663}
issue counts: {}
note counts: {'padding_heavy_expected': 11523, 'split_reassigned': 140}


In [3]:
tile_df = pd.DataFrame(tile_rows)
manifest_df = pd.DataFrame(manifest_rows)
fault_df = pd.DataFrame(fault_rows)

summary_df = (tile_df.groupby('source_dataset')
    .agg(tiles=('source_dataset', 'size'),
         zero_fraction_median=('zero_fraction', 'median'),
         zero_fraction_max=('zero_fraction', 'max'),
         non_finite_tiles=('non_finite_count', lambda s: int((s > 0).sum())),
         all_zero_tiles=('all_zero', 'sum'))
    .sort_index())
summary_df

,tiles,zero_fraction_median,zero_fraction_max,non_finite_tiles,all_zero_tiles
source_dataset,,,,,
anomaly_detection,9031,0.000000,0.992586,0,0
burned_area,6660,0.890625,0.890625,0,0
fire,1600,0.000000,0.000000,0,0
lc,63,0.781250,0.781250,0,0
roads,4800,0.781250,0.807211,0,0
worldfloods,2237,0.000000,0.999966,0,0


In [4]:
status_df = (tile_df.groupby(['source_dataset', 'status']).size().unstack(fill_value=0).sort_index())
status_df

status,ok,warn
source_dataset,,
anomaly_detection,9031,0
burned_area,0,6660
fire,1460,140
lc,0,63
roads,0,4800
worldfloods,2237,0


In [5]:
issue_df = (tile_df.explode('issue_codes')
    .dropna(subset=['issue_codes'])
    .groupby(['source_dataset', 'issue_codes']).size()
    .unstack(fill_value=0)
    .sort_index())
note_df = (tile_df.explode('note_codes')
    .dropna(subset=['note_codes'])
    .groupby(['source_dataset', 'note_codes']).size()
    .unstack(fill_value=0)
    .sort_index())

print('Issue counts by dataset')
display(issue_df if not issue_df.empty else pd.DataFrame())
print('Note counts by dataset')
display(note_df if not note_df.empty else pd.DataFrame())

Issue counts by dataset


""


Note counts by dataset


note_codes,padding_heavy_expected,split_reassigned
source_dataset,,
burned_area,6660,0
fire,0,140
lc,63,0
roads,4800,0


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
status_df.plot(kind='bar', stacked=True, ax=axes[0], color={'ok': '#2e8b57', 'warn': '#d49400', 'error': '#c0392b'})
axes[0].set_title('Tile status by dataset')
axes[0].set_ylabel('tiles')
axes[0].tick_params(axis='x', rotation=25)

plot_df = tile_df[['source_dataset', 'zero_fraction']].copy()
plot_df.boxplot(by='source_dataset', column='zero_fraction', ax=axes[1], grid=False)
axes[1].set_title('Zero fraction by dataset')
axes[1].set_ylabel('zero fraction')
axes[1].tick_params(axis='x', rotation=25)
fig.suptitle('')
plt.tight_layout()
plt.show()

In [7]:
plot_paths = summary['plot_paths']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, key, title in [
    (axes[0, 0], 'status_counts', 'Audit status plot'),
    (axes[0, 1], 'zero_fraction_by_expert', 'Zero fraction plot'),
    (axes[1, 0], 'representative_rgb_false_rgb', 'Representative RGB / false RGB'),
    (axes[1, 1], 'fault_rgb_false_rgb', 'Quarantined fault RGB / false RGB'),
]:
    image_path = Path(plot_paths.get(key, ''))
    ax.set_title(title)
    ax.axis('off')
    if image_path.exists():
        ax.imshow(mpimg.imread(image_path))
    else:
        ax.text(0.5, 0.5, f'missing: {image_path.name}', ha='center', va='center')
plt.tight_layout()
plt.show()

In [8]:
hard_errors = tile_df[tile_df['status'] == 'error'].copy()
warns = tile_df[tile_df['status'] == 'warn'].copy()
print('hard error rows:', len(hard_errors))
print('warn rows:', len(warns))
if len(hard_errors):
    display(hard_errors[['source_dataset', 'dataset_split', 'source_sample_id', 'materialized_image_path', 'issue_codes']].head(20))
else:
    print('No hard materialized-tile failures found.')

warns[['source_dataset', 'dataset_split', 'source_sample_id', 'materialized_image_path', 'note_codes']].head(20)

hard error rows: 0
warn rows: 11663
No hard materialized-tile failures found.


,source_dataset,dataset_split,source_sample_id,materialized_image_path,note_codes
9031,burned_area,train,0000002,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9032,burned_area,train,0000002,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9033,burned_area,train,0000002,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9034,burned_area,train,0000002,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9035,burned_area,train,0000002,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9036,burned_area,train,0000034,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9037,burned_area,train,0000034,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9038,burned_area,train,0000034,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9039,burned_area,train,0000034,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]
9040,burned_area,train,0000034,outputs/routerset/fix27March/images/burned_are...,[padding_heavy_expected]


In [9]:
fire_reassigned = tile_df[tile_df['note_codes'].apply(lambda codes: 'split_reassigned' in codes)].copy()
fire_reassigned = fire_reassigned[fire_reassigned['source_dataset'] == 'fire']
print('fire split-reassigned rows:', len(fire_reassigned))
fire_reassigned[['source_sample_id', 'source_split', 'dataset_split', 'materialized_image_path']].head(20)

fire split-reassigned rows: 140


,source_sample_id,source_split,dataset_split,materialized_image_path
16591,0002292,train,validation,outputs/routerset/fix27March/images/fire/train...
16592,0002293,train,validation,outputs/routerset/fix27March/images/fire/train...
16593,0002294,train,validation,outputs/routerset/fix27March/images/fire/train...
16594,0002295,train,validation,outputs/routerset/fix27March/images/fire/train...
16595,0002296,train,validation,outputs/routerset/fix27March/images/fire/train...
16596,0002297,train,validation,outputs/routerset/fix27March/images/fire/train...
16597,0002298,train,validation,outputs/routerset/fix27March/images/fire/train...
16598,0002299,train,validation,outputs/routerset/fix27March/images/fire/train...
16599,0002300,train,validation,outputs/routerset/fix27March/images/fire/train...
16600,0002302,train,validation,outputs/routerset/fix27March/images/fire/train...


In [10]:
fault_by_dataset = fault_df.groupby('source_dataset').size().sort_values(ascending=False)
fault_by_sample = (fault_df.groupby(['source_dataset', 'source_sample_id']).size()
    .sort_values(ascending=False)
    .rename('quarantined_tiles')
    .reset_index())

print('Quarantined tiles by dataset')
display(fault_by_dataset.to_frame('quarantined_tiles'))
print('Top concentrated source faults')
display(fault_by_sample.head(20))

Quarantined tiles by dataset


,quarantined_tiles
source_dataset,
anomaly_detection,185
worldfloods,116


Top concentrated source faults


,source_dataset,source_sample_id,quarantined_tiles
0,anomaly_detection,0000034,96
1,anomaly_detection,0000022,89
2,worldfloods,0009867,2
3,worldfloods,0023265,2
4,worldfloods,0028764,2
5,worldfloods,0012852,2
6,worldfloods,0013196,2
7,worldfloods,0024684,2
8,worldfloods,0014080,2
9,worldfloods,0014089,2


In [11]:
def inspect_tiles(dataset: str | None = None, status: str | None = None, limit: int = 6):
    subset = tile_df.copy()
    if dataset is not None:
        subset = subset[subset['source_dataset'] == dataset]
    if status is not None:
        subset = subset[subset['status'] == status]
    subset = subset.head(limit)
    if subset.empty:
        print('No rows matched.')
        return

    fig, axes = plt.subplots(len(subset), 2, figsize=(10, 4 * len(subset)))
    if len(subset) == 1:
        axes = np.array([axes])
    for row_axes, (_, row) in zip(axes, subset.iterrows()):
        tile = np.load(row.get('resolved_materialized_image_path') or row['materialized_image_path'])
        rgb = np.stack([tile[2], tile[1], tile[0]], axis=-1)
        frgb = np.stack([tile[4], tile[2], tile[1]], axis=-1)
        for ax, image, title in [(row_axes[0], rgb, 'RGB'), (row_axes[1], frgb, 'False RGB')]:
            low, high = np.quantile(image, [0.02, 0.98])
            if not np.isfinite(low) or not np.isfinite(high) or high <= low:
                high = max(float(np.max(image)), 1.0)
                low = min(float(np.min(image)), 0.0)
            image = np.clip((image - low) / max(high - low, 1e-6), 0.0, 1.0)
            ax.imshow(image)
            ax.set_title(f"{row['source_dataset']} / {row['dataset_split']} / {title}")
            ax.axis('off')
        print(row['materialized_image_path'])
        print('  status=', row['status'], 'issues=', row['issue_codes'], 'notes=', row['note_codes'])
        print('  zero_fraction=', round(float(row['zero_fraction']), 4), 'min/max=', (float(row['min']), float(row['max'])))
    plt.tight_layout()
    plt.show()


inspect_tiles('roads', 'warn', limit=3)

outputs/routerset/fix27March/images/roads/train/500shot_train_000000_0_0_256_256.npy
  status= warn issues= [] notes= ['padding_heavy_expected']
  zero_fraction= 0.7812 min/max= (0.0, 0.4108999967575073)


outputs/routerset/fix27March/images/roads/train/500shot_train_000001_0_0_256_256.npy
  status= warn issues= [] notes= ['padding_heavy_expected']
  zero_fraction= 0.7812 min/max= (0.0, 0.33410000801086426)


outputs/routerset/fix27March/images/roads/train/500shot_train_000002_0_0_256_256.npy
  status= warn issues= [] notes= ['padding_heavy_expected']
  zero_fraction= 0.7812 min/max= (0.0, 0.5848000049591064)


## Reading the audit

- `tile_audit.jsonl` is the file-by-file audit record. Each row corresponds to one materialized `.npy` tile.
- `status=error` means the exported tile itself is broken, for example missing, wrong shape, wrong dtype, non-finite, or all-zero.
- `status=warn` is used for expected-but-important conditions, mainly heavy zero padding on `roads`, `lc`, and `burned_area`, and the rebuilt `fire` validation split.
- `fault_rows_256.jsonl` is the quarantine list for tiles excluded from the clean export.
